# Measuring the Redshift of a JWST/NIRSpec Spectrum

This tutorial fits the redshift of a real JWST/NIRSpec spectrum with BESTA's `SpectraRedshiftFitModule` using a full-spectrum likelihood and a grid search over redshift.

## What this tutorial covers

- Loading a one-dimensional NIRSpec spectrum from a FITS file
- Masking unusable pixels and exporting the spectrum to the text format expected by the pipeline
- Configuring and running a BESTA redshift fit
- Reading the maximum-likelihood solution and comparing it to a reference redshift
- Plotting the best-fit and residuals for a quick quality check

## Before you run it

- Make sure that `besta`, `astropy`, and `matplotlib` libraries are available in the current python kernel


In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from astropy import units as u
from astropy.io import fits

## 1. Input Spectrum

The data used for this tutorial was downloaded from the **[DAWN JWST archive](https://dawn-cph.github.io/dja/)**.

The FITS files contain the extracted one-dimensional spectrum, its uncertainty, and a sky estimate. This first pass is only for inspection: we plot the raw arrays and build a simple binary mask that downweights invalid pixels before fitting.

Pixels with non-finite fluxes or non-positive uncertainties are treated as unusable. The notebook keeps them in the exported file but assigns them a zero weight through a separate mask file.

In [ ]:
tutorial_dir = Path.cwd()
# A post-starburst galaxy without emission lines
file = "valentino-cosmos04-v4_g235m-f170lp_3567_54459.spec.fits"
z_true = 3.7129
# A starburst galaxy with strong emission lines
# file = "abell2744-glass-v4_g140h-f100lp_1324_20021.spec.fits"
# z_true = 1.3668
spectrum_path = tutorial_dir / file

if not spectrum_path.exists():
    raise FileNotFoundError(f"Could not find the tutorial spectrum at {spectrum_path}")

with fits.open(spectrum_path) as hdul:

    wl = hdul["SPEC1D"].data["wave"] * u.micron
    flux = hdul["SPEC1D"].data["flux"] * u.uJy
    flux_error = hdul["SPEC1D"].data["err"] * u.uJy
    sky = hdul["SPEC1D"].data["sky"]

plt.figure(figsize=(10, 5))
plt.plot(wl, flux, label="Flux")
plt.plot(wl, sky, label="Sky")
plt.plot(wl, flux_error, label="Flux Error")
plt.xlabel("Wavelength (micron)")
plt.ylabel("Flux (micro Jy)")
plt.title("JWST NIRSpec Spectrum")
plt.yscale("log")
plt.legend()
plt.show()

bad_pixels = ~np.isfinite(flux) | ~np.isfinite(flux_error) | (flux_error <= 0) | (wl < 1.0 * u.micron) | (wl > 5.0 * u.micron)
weights = 1 - bad_pixels.astype(float)
flux_error[bad_pixels] = 1e10 * u.uJy


## 2. Convert to the Flux Units Used by BESTA

The pipeline expects specific flux density per wavelength unit. If no units are specified, BESTA assumes the input spectra is in $10^{-16}\,\mathrm{erg\ s^{-1}\ cm^{-2}\ \AA^{-1}}$.

In [ ]:
# Prepare the data for fitting

flux_flam = (flux * (3e8 * u.km / u.s) / wl**2).to(u.erg / (u.s * u.cm**2 * u.AA))

flux_error_flam = (flux_error * (3e8 * u.km / u.s) / wl**2).to(u.erg / (u.s * u.cm**2 * u.AA))

spectrum_txt_path = tutorial_dir / "jwst_spectrum.txt"
weights_txt_path = tutorial_dir / "jwst_spectrum_weights.txt"

plt.figure(figsize=(10, 5))
plt.semilogy(wl, flux_flam, label="Flux (erg/s/cm^2/AA)")

plt.xlabel("Wavelength (micron)")
plt.ylabel("Flux (erg/s/cm^2/AA)")
plt.title("JWST NIRSpec Spectrum (Flam)")
plt.legend()

np.savetxt(
    spectrum_txt_path,
    np.column_stack((
        wl.to_value(u.AA),
        flux_flam.to_value(u.erg / (u.s * u.cm**2 * u.AA)),
        flux_error_flam.to_value(u.erg / (u.s * u.cm**2 * u.AA)),
    )),
    header="Wavelength (micron) Flux (erg/s/cm^2/AA) Flux_Error (erg/s/cm^2/AA)",
)

np.savetxt(
    weights_txt_path,
    weights,
    header="Weights for fitting (1=good pixel, 0=bad pixel)",
)

### (Optional) pre-build the SSP model to speed up

If you want to infer the redshift from more than one spectrum, it is highly recommended to first pre-build the SSP stellar template to the desired resolution.

In this case, we will resample the Bruzual & Charlote 2003 (version updated in 2016) SSP models to a resolution of constant velocity spanning the entire range available. The SSP grid can be saved as a pickle file, that can be loaded on each run, skipping the interpolation step.

**Security disclaimer**: Python pickle files can execute arbitrary code when loaded. Only load .pkl files that you generated yourself or received from a trusted source. Never load pickle files from untrusted or unverified locations.

In [ ]:
from pst.SSP import BC03_2016

stelib_bc3 = BC03_2016(model="stelib", imf="cha", gas_emission=True)

vel_scale = 300  # km / s
log_wl_model = np.log(stelib_bc3.wavelength.to_value(u.AA))
log_wl_obs = np.log(wl.to_value(u.AA))
log_wl_grid = np.arange(log_wl_model[0], log_wl_model[-1], vel_scale / 3e5)
log_wl_obs_grid = np.arange(log_wl_obs[0], log_wl_obs[-1], vel_scale / 3e5)

stelib_bc3.interpolate_sed(np.exp(log_wl_grid) << u.AA)

pickle_path = tutorial_dir / "stelib_bc03.pkl"
stelib_bc3.to_pickle(pickle_path)

## 3. Configure and Run the Redshift Fit

The next cells import the BESTA pipeline manager and define a minimal configuration for a maximum-likelihood redshift fit.

For redshift determination, BESTA currently provides the module **`SpectraRedshiftFitModule`**. When using this module, include the following setup details:

- The section `pipeline/extra_output` must include `redshift/redshift`.
- The search interval is controlled by `z_min` and `z_max`.
- Keep the observed spectrum in the observed frame; the module shifts the model internally during the scan.

In this tutorial we also enable `use_features="T"`, which estimates a smooth continuum and feature-sensitive weights. This usually improves robustness when broad-band continuum shape mismatches dominate over line information. In poor SNR conditions, it is recommended to disable it.

In [ ]:
from besta import MainPipeline
from besta.pipeline_modules.spectra_redshift_fit import SpectraRedshiftFitModule

In [ ]:
output_root = tutorial_dir / "results.jwst_redshift_maxlike.txt"
values_path = tutorial_dir / "tutorial_values.ini"

configuration = {
    "runtime": {
        "sampler": "grid",
    },
    "emcee": {
        "nwalkers": 16,
        "samples": 50,
    },
    "maxlike": {
        "method": "Powell",
        "tolerance": 1e-6,
        "maxiter": 500,
    },
    "grid": {
        "nsample_dimension": 4,
    },
    "snake": {
        "threshold": 1.0,
        "nsample_dimension": 10,
        "maxiter": 5000,
    },
    "output": {
        "filename": str(output_root),
        "format": "text",
    },
    "pipeline": {
        "modules": "SpectraRedshiftFit",
        "values": str(values_path),
        "likelihoods": "SpectraRedshiftFit",
        "timing": "T",
        # IMPORTANT: always include this when using SpectraRedshiftFitModule
        "extra_output": "redshift/redshift",
    },
    "SpectraRedshiftFit": {
        "file": SpectraRedshiftFitModule.get_path(),
        "inputSpectrum": str(spectrum_txt_path),
        "mask": str(weights_txt_path),
        # If using the pickled model
        "SSPModelFromPickle": str(pickle_path),
        # If not using the pickled model, specify the model and args as usual.
        "SSPModel": "BC03_2016",
        "SSPModelArgs": "stelib, cha, gas_emission=True",
        "SSPDir": "None",
        "velscale": 300.0,
        "z_max": 5.0,
        "z_min": 0.0,
        "use_features": "T",
        "continuum_knot_spacing": 300.0,
        "continuum_sigma_clip": 3.0,
        "logging_console": "T",
        "SFHModel": "DelayedTauQuenchedSFH",
        "save_z_loglike": str(tutorial_dir / "jwst_redshift_loglike.txt"),
    },
}

# Quick configuration sanity checks before running the pipeline
assert configuration["SpectraRedshiftFit"]["z_min"] < configuration["SpectraRedshiftFit"]["z_max"]
assert Path(configuration["SpectraRedshiftFit"]["inputSpectrum"]).exists(), "Input spectrum file not found"
assert Path(configuration["SpectraRedshiftFit"]["mask"]).exists(), "Mask file not found"

### Configuring the model priors

The small `values.ini` file only exposes a few nuisance parameters so the tutorial stays fast and the optimizer has a valid search region.

In [ ]:
# Redshift is inferred internally by the module. Keep the SFH mostly fixed, but
# allow a modest logtau range so maxlike has a valid search space.
if values_path.exists():
    values_path.unlink()

with open(values_path, "w") as f:
    f.write("[stars.sfh]\n")
    f.write("logtau = 0 0.5 1.5\n")
    f.write("quenching_time = 13.6 13.8 13.8\n")
    f.write("alpha_powerlaw = 1\n")
    f.write("ism_metallicity_today = 0.005 0.02 0.05\n")
    f.write("[dust.extinction]\n")
    f.write("a_v = 0.0 0.1  1.0\n")


### Run the pipeline

Now we can use the pipeline manager to run the module. By setting `plot_result=True`, BESTA will generate a plot comparing the best-fit model and the input spectra.

In [ ]:
pipeline = MainPipeline(pipeline_configuration_list=[configuration])
exit_code = pipeline.execute_all(plot_result=True)
if exit_code:
    raise RuntimeError("Pipeline exited with a non-zero code. Check the logs.")

## 4. Inspect the Solution

After the pipeline finishes, the `Reader` helper loads the output text file and reconstructs the maximum-likelihood parameter block. We then re-run the last module once so that derived arrays, such as the model spectrum on the observed wavelength grid, are available for analysis and plotting.

This section compares the recovered value against the reference redshift used at the top of the notebook (`z_true`).

In [ ]:
from besta import Reader

reader = Reader.from_results_file(str(output_root))
reader.load_results()
solution = reader.get_maxlike_solution(as_datablock=True)
module = reader.last_module
module.execute(solution)

z_recovered = solution["redshift", "redshift"]
print(f"\nRecovered redshift: {z_recovered:.4f}")
print(f"True redshift: {z_true:.4f}")
print(f"Redshift error: {abs(z_recovered - z_true):.4f}")


## 5. Compare the Best-Fit Model to the Spectrum

First, let's examine the likelihood distribution across redshift values. The module exposes the `z_loglike` array containing the log-likelihood evaluated at each redshift step.

In [ ]:
# Normalize the log-likelihood profile for better visualization
z_like_norm = module.z_loglike - module.z_loglike.max()

# Get the 100 best-fitting redshift steps (those with the highest log-likelihood values)
best_loglikes = np.sort(z_like_norm)[-100:]
best_loglike_index = np.argsort(z_like_norm)[-100:]
best_redshifts = module.config["slice_redshifts"][best_loglike_index]

plt.plot(module.config["slice_redshifts"], z_like_norm, "-o")
plt.axvline(z_true, color="k", ls="--", label=f"True z = {z_true:.4f}")
plt.ylim(best_loglikes.min(), best_loglikes.max())
plt.xlim(best_redshifts.min() - 0.1, best_redshifts.max() + 0.1)
plt.legend()
plt.xlabel("Redshift")
plt.ylabel("Delta Log-Likelihood")
plt.title("Redshift Log-Likelihood Profile")
plt.show()

Now let us make a final plot comparing the observed spectrum and the best-fit model

In [ ]:
wl = module.config["wavelength"].value   # Angstrom
obs = module.config["flux"]
err = np.sqrt(module.config["var"])
flux_model, _ = module.make_observable(solution, parse=True)

plot_path = tutorial_dir / "redshift_fit_full_sampling_jwst.png"

fig, axes = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

axes[0].fill_between(wl, obs - err, obs + err, color="k", alpha=0.2)
axes[0].plot(wl, obs, color="k", lw=0.8, label="JWST observed")
axes[0].plot(
    wl, flux_model, color="C1", lw=1.2,
    label=f"Best-fit  z = {z_recovered:.4f}",
)

axes[0].set_ylabel("Flux")
axes[0].legend()
axes[0].set_title("JWST redshift full-sampling demo  –  MaxLike")
axes[0].set_xlim(wl.min(), wl.max())
axes[0].set_ylim(flux_model.min(), 2 * flux_model.max())

residuals = (obs - flux_model) / err
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].fill_between(wl, -1, 1, color="k", alpha=0.1, label="±1 σ")
axes[1].plot(wl, residuals, color="C0", lw=0.6)
axes[1].set_ylabel("Residuals")
axes[1].set_xlabel("Wavelength [Å]")
axes[1].set_ylim(-5, 5)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(plot_path, dpi=150)
print(f"Plot saved to {plot_path}")
plt.show()

### Feature-weight diagnostics

With `use_features="T"`, the module builds a smooth continuum estimate and a feature-sensitive weight, combined with user-defined weights as:

$w=\left(\frac{f - c}{\sigma(c)}\right)^2\cdot w_{\rm usr}$ 

where $c$ denotes the continuum.

This way, strong emission lines or deep absorption features receive higher weight during the fit, at the expense of flat or feature-less spectral regions, which would otherwise lead to high redshift degeneracy.

- Peaks in the weight curve indicate spectral regions that carry stronger redshift information.
- Near-zero weights indicate regions downweighted by masking or low-information continuum.
- Large continuum uncertainties usually correspond to lower-confidence regions.


In [ ]:
if configuration["SpectraRedshiftFit"]["use_features"] == "T":
    continuum = module.config["continuum"]
    continuum_err = module.config["continuum_err"]
    weights = module.config["sweep_weights"]

    fig, axs = plt.subplots(2, 1, figsize=(10, 6), constrained_layout=True)
    axs[0].plot(wl, obs, color="k", lw=0.8, label="JWST observed")
    axs[0].fill_between(wl, obs - err, obs + err, color="k", alpha=0.2)
    axs[0].plot(wl, continuum, color="C1", lw=1.2, label="Continuum")
    axs[0].fill_between(
        wl, continuum - continuum_err, continuum + continuum_err,
        color="C1", alpha=0.3, label="Continuum Error"
    )
    axs[0].set_xlabel("Wavelength [Å]")
    axs[0].set_ylabel("Flux")
    axs[0].set_title("Continuum and Feature-Based Weights")
    axs[0].legend()
    axs[0].set_xlim(wl.min(), wl.max())
    axs[0].set_ylim(continuum.min(), continuum.max() * 3)

    axs[1].plot(wl, weights, color="C2", lw=1.2, label="Feature-Based Weights")
    axs[1].set_yscale("symlog", linthresh=0.01)
    axs[1].axhline(0, color="k", lw=0.8, ls="--")
    axs[1].set_xlabel("Wavelength [Å]")
    axs[1].set_ylabel("Weight")
    axs[1].set_title("Feature-Based Weights for Redshift Fitting")
    axs[1].legend()
    axs[1].set_xlim(wl.min(), wl.max())
    plt.show()